In [96]:
import cv2
import numpy as np
# cv2.VideoCapture 视频抽帧，视频图像化
# 参数是视频文件路径则打开
cap = cv2.VideoCapture('D:/test.avi')
all_frames = []
sum_frames = np.zeros((576,768,3))
i = 0
while(cap.isOpened()):
	# ret是布尔值，如果读取帧正确返回true，当读到结尾会返回false
	# frame是每一帧的图像，是个三维矩阵
    ret, frame = cap.read()
    if not ret :
        break
    #frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    all_frames.append(frame)
    sum_frames+=frame
    i+=1
cap.release()
cv2.destroyAllWindows()


In [97]:
mean_video = sum_frames / i


In [100]:
cv2.imshow("img",all_frames[0])
cv2.waitKey(0)
cv2.destroyAllWindows()

In [101]:
to_train = []
for i in all_frames:
    i = i - mean_video
    to_train.append(i)

In [99]:
cv2.imshow("img",mean_video)
cv2.waitKey(0)
cv2.destroyAllWindows()

In [102]:
for i in to_train:
    cv2.imshow('video',i)
    if cv2.waitKey(25) & 0xFF == 27: break
cv2.waitKey(0)

-1

In [30]:
x = []
to_train = np.array(to_train)
to_train = to_train.astype(np.float64)

In [93]:
x = []
for i in to_train:
    i = cv2.blur(i,(3,3))
    x.append(i)
for i in x:
    cv2.imshow('video',i)
    if cv2.waitKey(25) & 0xFF == 27: break
cv2.waitKey(0)

-1

In [60]:
y = []
kernel = np.ones((3,3), np.uint8)
for i in x:
    i=cv2.erode(i,kernel,iterations=1)
    y.append(i)
for i in y:
    cv2.imshow('video',i)
    if cv2.waitKey(25) & 0xFF == 27: break
cv2.waitKey(0)


-1

In [115]:
import cv2 as cv
import numpy as np

file_test = "D:/test.avi"
cap = cv.VideoCapture(file_test)
kernel = cv.getStructuringElement(cv.MORPH_ELLIPSE, (2, 2))
color_m = (255, 0, 0)
bg=cv.createBackgroundSubtractorMOG2()    #背景建模做差
out_fps = 12.0
fourcc = cv.VideoWriter_fourcc('M', 'P', '4', '2')
out = cv.VideoWriter('D:/testa.avi', fourcc, out_fps, (500, 500))

while True:
    ret, frame = cap.read()
    if not ret:
        break
    frame_motion = frame.copy()
    fgmask = bg.apply(frame_motion)
    draw1 = cv.threshold(fgmask, 25, 255, cv.THRESH_BINARY)[1]  #二值化
    draw1 = cv.dilate(draw1, kernel, iterations=1)  #膨胀
    draw1 = cv.erode(draw1, kernel, iterations=1)   #腐蚀
    contours_m, hierarchy_m = cv.findContours(draw1.copy(), cv.RETR_EXTERNAL, cv.CHAIN_APPROX_SIMPLE)   #做框
    for c in contours_m:
        if cv.contourArea(c) < 300:
            continue
        (x, y, w, h) = cv.boundingRect(c)
        cv.rectangle(frame_motion, (x, y), (x + w, y + h), color_m, 2)
    cv.imshow("source", frame_motion)
    cv.imshow("draw", draw1)
    k = cv.waitKey(25)
    if k == ord('q'):
        break

    out.write(frame_motion)

out.release()
cap.release()
cv.destroyAllWindows()